# Time-to-event DeepHit postprocessing

This notebook discovers completed artifacts from the streaming time-to-event sweep, selects the best configuration in each model family, and packages the selected artifacts for downstream figure generation.

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

result_folder = 'results_official_experiment'

baseline_result_folder = 'results_ctg_baselines_1hz'

# Optional outputs from postprocess_end_of_tracing_trained_classifier.ipynb.
INCLUDE_PRETRAINED_LAST_HOUR_CLASSIFIER = True
pretrained_classifier_result_folder = 'results_classifier_last_hour'
pretrained_classifier_stem = 'Inception_classifier_last_hour'

REPO_ROOT = Path.cwd().resolve()
FINAL_SELECTION_DIR = REPO_ROOT / 'final_model_selection'
RESULT_DIR = REPO_ROOT / result_folder
PRETRAINED_CLASSIFIER_RESULT_DIR = REPO_ROOT / pretrained_classifier_result_folder
EXCLUDED_RESULT_DIRS = {
    baseline_result_folder,
    pretrained_classifier_result_folder,
    'results_ctu_checkpoint_smoke',
}
RESULT_DIRS = [RESULT_DIR]
missing_result_dirs = [path for path in RESULT_DIRS if not path.is_dir()]
if missing_result_dirs:
    raise FileNotFoundError(f'Missing result directories: {missing_result_dirs}')
BASELINE_RESULT_DIR = REPO_ROOT / baseline_result_folder
OUTPUT_DIR = REPO_ROOT / result_folder / 'postprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change these as desired. Endpoint rows are retained for the selected model configurations.
SPLIT = 'Test'
ROC_END_MINUTES = 0.0
# The ROC panel uses the highest-scoring selected configuration by default.
# Set this to a raw family such as 'marked' to restrict that choice.
ROC_FAMILY = None
# Keep one architecture/input configuration per model-selection family.
# 'best in family' selects the highest reduced score on the configured target/metric.
MODEL_SELECTION = 'best in family'
MODEL_SELECTION_SPLIT = 'Validation'
MODEL_SELECTION_TARGET = 'macro_pH'
# Examples: 'dynamic_auc', 'antolini_cindex', 'auroc_end_landmark', 'auroc_acidemic_delivery_by_horizon'
MODEL_SELECTION_METRIC = 'auroc_acidemic_delivery_by_horizon'
# Reduce all selected metric rows for a configuration with this operation.
MODEL_SELECTION_REDUCTION = 'mean'  # also accepts 'median'
# Include end-relative AUROC rows alongside MODEL_SELECTION_METRIC.
INCLUDE_END_LANDMARKS_IN_SELECTION = True

# Columns that define the groups in which model selection is performed.
# Sampling regime, depth, and kernel size remain candidate dimensions.
# Use ['family', 'training_regime'] to select separately per family and
# sampling regime, or ['training_regime'] to select one model per regime.
MODEL_SELECTION_FAMILY_COLUMNS = ['family']

FAMILY_LABEL = {
    # 'survival': 'Threshold DeepHit',
    'marked': 'Marked DeepHit',
    # 'marked_corn': 'Marked DeepHit (CORN)',
    'competing': 'Competing-risk DeepHit',
    # 'competing_threshold': 'Binary competing-risk DeepHit',
    'baseline_ctg_only_fri_cox': 'CTG-only FRI Cox',
    'baseline_xgboost_figo_ctg_cox': 'XGBoost FIGO CTG Cox',
    'classifier_inception_last_hour': 'InceptionTime (last-hour classifier)',
}
HEAD_LABEL = {
    'joint': 'Shared head',
    'cause_specific': 'Independent head',
    'not_applicable': '',
}
TRAINING_REGIME_LABEL = {
    'all': 'All',
    'quartiles_plus_terminal': 'Quartiles + terminal',
    'terminal_balanced': 'Terminal balanced',
    'stratified_elapsed': 'Stratified elapsed',
    'uniform': 'Uniform',
    'unknown': 'Training regime unavailable',
    'pretrained_classifier': 'Pretrained classifier',
    'not_applicable': 'Baseline',
}
DISPLAY_FAMILY_ORDER = [
    'CTG-only FRI Cox',
    'XGBoost FIGO CTG Cox',
    'InceptionTime (last-hour classifier)',
    # 'Threshold DeepHit',
    'Marked DeepHit',
    # 'Marked DeepHit (CORN)',
    # 'Competing-risk DeepHit — Shared head',
    'Competing-risk DeepHit — Independent head',
    # 'Binary competing-risk DeepHit — Shared head',
    'Binary competing-risk DeepHit — Independent head',
]

def display_family_label(family: str, head_type: str) -> str:
    base_label = FAMILY_LABEL.get(family, family)
    if family in {'competing', 'competing_threshold'}:
        return f'{base_label} — {HEAD_LABEL.get(head_type, head_type)}'
    return base_label

IMPUTATION_LABEL = {
    'ffill': 'Forward fill',
    'none': 'Native missing-value mask',
}
MODEL_LABEL = {
    'inception': 'InceptionTime',
    'inceptiontime-lstm': 'InceptionTime-LSTM',
    'inception-lstm': 'InceptionTime-LSTM (8h history)',
    'inception-classifier': 'InceptionTime classifier',
}
# Compare all complete model/input configurations by default.
# Use 'model_type' without an imputation filter to show every available
# model/input pairing. Set an imputation filter only for a matched comparison.
# COMPARE_BY = 'configuration'  # 'configuration', 'imputation', 'model_type', or None
# COMPARE_BY = 'imputation'
COMPARE_BY = 'model_type'
# COMPARE_BY = None
MODEL_TYPE_FILTER = None  # e.g. 'inceptiontime-lstm', 'inception-lstm'
IMPUTATION_FILTER = None  # e.g. 'ffill', 'none'
# Set False to retain the marked-CORN ordinal ablation.
EXCLUDE_CORN_RESULTS = True
if COMPARE_BY not in {'configuration', 'imputation', 'model_type', None}:
    raise ValueError(f'Unsupported COMPARE_BY: {COMPARE_BY!r}')
if MODEL_TYPE_FILTER is not None and MODEL_TYPE_FILTER not in MODEL_LABEL:
    raise ValueError(f'Unknown MODEL_TYPE_FILTER: {MODEL_TYPE_FILTER!r}')
if IMPUTATION_FILTER is not None and IMPUTATION_FILTER not in IMPUTATION_LABEL:
    raise ValueError(f'Unknown IMPUTATION_FILTER: {IMPUTATION_FILTER!r}')
if not isinstance(EXCLUDE_CORN_RESULTS, bool):
    raise TypeError('EXCLUDE_CORN_RESULTS must be boolean')
if MODEL_SELECTION not in {'best in family', 'all'}:
    raise ValueError(f'Unsupported MODEL_SELECTION: {MODEL_SELECTION!r}')
if MODEL_SELECTION_SPLIT not in {'Train', 'Validation', 'Test'}:
    raise ValueError(f'Unsupported MODEL_SELECTION_SPLIT: {MODEL_SELECTION_SPLIT!r}')
if MODEL_SELECTION_REDUCTION not in {'mean', 'median'}:
    raise ValueError(f'Unsupported MODEL_SELECTION_REDUCTION: {MODEL_SELECTION_REDUCTION!r}')
if not MODEL_SELECTION_FAMILY_COLUMNS:
    raise ValueError('Specify at least one model-selection family column.')
if len(MODEL_SELECTION_FAMILY_COLUMNS) != len(set(MODEL_SELECTION_FAMILY_COLUMNS)):
    raise ValueError('MODEL_SELECTION_FAMILY_COLUMNS must not contain duplicates.')
if 'model_configuration' in MODEL_SELECTION_FAMILY_COLUMNS:
    raise ValueError('model_configuration is selected within a family and cannot define the family.')
if ROC_FAMILY is not None and ROC_FAMILY not in FAMILY_LABEL:
    raise ValueError(f'Unknown ROC_FAMILY: {ROC_FAMILY!r}')
if COMPARE_BY == 'configuration':
    if MODEL_TYPE_FILTER is not None or IMPUTATION_FILTER is not None:
        raise ValueError('Comparing all configurations requires no model or imputation filter.')
    COMPARISON_COLUMN, COMPARISON_TITLE = 'configuration_label', 'Model / input configuration / training regime'
elif COMPARE_BY == 'imputation':
    if MODEL_TYPE_FILTER is None or IMPUTATION_FILTER is not None:
        raise ValueError('Comparing imputation requires one MODEL_TYPE_FILTER and no IMPUTATION_FILTER.')
    COMPARISON_COLUMN, COMPARISON_TITLE = 'imputation_label', 'Imputation method / training regime'
elif COMPARE_BY == 'model_type':
    if MODEL_TYPE_FILTER is not None:
        raise ValueError('Comparing model types requires no model filter.')
    if IMPUTATION_FILTER is None:
        COMPARISON_COLUMN, COMPARISON_TITLE = 'model_label', 'Model type / training regime (pooled across input methods)'
    else:
        COMPARISON_COLUMN, COMPARISON_TITLE = 'model_label', 'Model type / training regime'
else:
    if MODEL_TYPE_FILTER is None or IMPUTATION_FILTER is None:
        raise ValueError('COMPARE_BY=None requires both MODEL_TYPE_FILTER and IMPUTATION_FILTER.')
    COMPARISON_COLUMN, COMPARISON_TITLE = None, 'Fixed configuration / training regime'
POOL_INPUT_METHODS = COMPARE_BY == 'model_type' and IMPUTATION_FILTER is None
if not RESULT_DIR.exists():
    raise FileNotFoundError(f'Missing results directory: {RESULT_DIR}')
print(f'Reading results from {RESULT_DIR}')
if POOL_INPUT_METHODS:
    print('Pooling available input methods within each model type for this comparison.')

In [ ]:
RUN_RE = re.compile(r'^(competing_threshold|survival|marked_corn|marked|competing)_deephit_(.+)$')
MODEL_CLASS_TYPE = {
    'Inception_classifier': 'inception',
    'InceptionTime_LSTM_classifier': 'inceptiontime-lstm',
    'Inception_LSTM_deephit': 'inception-lstm',
}
MODEL_RUN_RE = re.compile(
    r'(?:^|_)(inceptiontime-lstm|inception-lstm-deephit|inception)(?:_|$)'
)
MODEL_RUN_TYPE = {
    'inceptiontime-lstm': 'inceptiontime-lstm',
    'inception-lstm-deephit': 'inception-lstm',
    'inception': 'inception',
}

def artifact_stem(path: Path) -> str:
    for suffix in ('_history.csv', '_landmark_metrics.csv', '_landmark_predictions.csv'):
        if path.name.endswith(suffix):
            return path.name.removesuffix(suffix)
    raise ValueError(f'Unrecognized run artifact: {path.name}')

def run_identity(path: Path) -> tuple[str, str, str]:
    stem = artifact_stem(path)
    match = RUN_RE.match(stem)
    if match is None:
        raise ValueError(f'Unrecognized run artifact: {path.name}')
    return match.group(1), match.group(2), stem

def manifest_metadata(result_dir: Path, stem: str) -> dict:
    path = result_dir / f'{stem}.json'
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text())
    except (OSError, json.JSONDecodeError) as error:
        print(f'Ignoring unreadable manifest {path.name}: {error}')
        return {}

def fallback_metadata(run_id: str) -> tuple[str, str]:
    imputation_match = re.search(r'(?:^|_)impute-([^_]+)', run_id)
    model_match = MODEL_RUN_RE.search(run_id)
    if imputation_match is None or model_match is None:
        raise ValueError(f'Missing model or imputation metadata in run ID: {run_id}')
    return MODEL_RUN_TYPE[model_match.group(1)], imputation_match.group(1)

def model_configuration_metadata(model_metadata: dict, run_id: str, model_type: str, imputation: str) -> tuple[str, str]:
    # fit_kwargs identify the depth/kernel variant while model type and
    # imputation prevent otherwise identical configurations from colliding.
    fit_kwargs = model_metadata.get('fit_kwargs') or {}
    if not isinstance(fit_kwargs, dict):
        fit_kwargs = {}
    configuration_key = json.dumps(
        {'model_type': model_type, 'imputation': imputation, 'fit_kwargs': fit_kwargs},
        sort_keys=True, default=str,
    )
    depth = fit_kwargs.get('depth')
    kernel_sizes = fit_kwargs.get('kernel_sizes')
    if depth is not None or kernel_sizes is not None:
        configuration_label = f'depth={depth}, kernels={kernel_sizes}'
    elif fit_kwargs:
        configuration_label = json.dumps(fit_kwargs, sort_keys=True, default=str)
    else:
        # Older artifacts without manifests cannot expose their fit kwargs.
        configuration_key = run_id
        configuration_label = run_id
    return configuration_key, configuration_label

def result_file_key(family: str, metadata: dict, run_id: str) -> tuple[str, ...]:
    model = metadata.get('model') or {}
    fit_kwargs = model.get('fit_kwargs') or {}
    seed = metadata.get('random_state')
    if seed is None:
        match = re.search(r'(?:^|_)s(\d+)(?:_|$)', run_id)
        seed = match.group(1) if match else 'unknown'
    return (family, str(fit_kwargs.get('depth')), json.dumps(fit_kwargs.get('kernel_sizes'), sort_keys=True, default=str), str(metadata.get('training_sampling_strategy')), str(seed))

def load_artifacts(suffix: str) -> pd.DataFrame:
    frames = []
    seen_result_files = set()
    for path in [
        path for result_dir in RESULT_DIRS for path in result_dir.glob(f'*{suffix}')
    ]:
        source_result_dir = path.parent
        try:
            family, run_id, stem = run_identity(path)
            if EXCLUDE_CORN_RESULTS and family == 'marked_corn':
                continue
            frame = pd.read_csv(path)
            metadata = manifest_metadata(source_result_dir, stem)
            model_metadata = metadata.get('model', {})
            model_class = model_metadata.get('model_class')
            imputation = metadata.get('imputation', {}).get('method')
            training_regime = metadata.get('training_sampling_strategy') or 'unknown'
            if family in {'competing', 'competing_threshold'}:
                head_type = model_metadata.get('competing_head') or 'joint'
            else:
                head_type = 'not_applicable'
            if model_class in MODEL_CLASS_TYPE and imputation:
                model_type = MODEL_CLASS_TYPE[model_class]
            else:
                model_type, imputation = fallback_metadata(run_id)
            model_configuration, model_configuration_label = model_configuration_metadata(
                model_metadata, run_id, model_type, imputation
            )
        except (ValueError, pd.errors.EmptyDataError) as error:
            print(f'Skipping {path.name}: {error}')
            continue

        selection_key = result_file_key(family, metadata, run_id)
        if selection_key in seen_result_files:
            print(f'Skipping duplicate result file (keeping first): {path}')
            continue
        seen_result_files.add(selection_key)

        # These are values that may be used to set a family 
        frame['family'] = family
        frame['family_base_label'] = FAMILY_LABEL.get(family, family)
        frame['head_type'] = head_type
        frame['head_label'] = HEAD_LABEL.get(head_type, head_type)
        frame['family_label'] = display_family_label(family, head_type)
        frame['training_regime'] = training_regime
        frame['training_regime_label'] = TRAINING_REGIME_LABEL.get(
            training_regime, str(training_regime).replace('_', ' ').title()
        )
        frame['imputation'] = imputation
        frame['imputation_label'] = IMPUTATION_LABEL.get(imputation, imputation)
        frame['model_type'] = model_type
        frame['model_label'] = MODEL_LABEL.get(model_type, model_type)
        frame['model_configuration'] = model_configuration
        frame['model_configuration_label'] = model_configuration_label
        frame['configuration_label'] = frame['model_label'] + ' — ' + frame['imputation_label']
        frame['run_id'] = run_id
        frame['source_file'] = path.name
        frame['source_path'] = str(path)
        frame['source_result_dir'] = str(source_result_dir)
        frame['source_relative_path'] = path.relative_to(REPO_ROOT).as_posix()
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

CLASSIFIER_FAMILY = 'classifier_inception_last_hour'

def load_pretrained_classifier_artifacts() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load optional landmark CSVs from the standalone classifier notebook."""
    empty = pd.DataFrame()
    if not INCLUDE_PRETRAINED_LAST_HOUR_CLASSIFIER:
        return empty, empty
    if not PRETRAINED_CLASSIFIER_RESULT_DIR.is_dir():
        print(f'Pretrained classifier result directory not found: {PRETRAINED_CLASSIFIER_RESULT_DIR}')
        return empty, empty
    manifest_path = PRETRAINED_CLASSIFIER_RESULT_DIR / f'{pretrained_classifier_stem}.json'
    metadata = {}
    if manifest_path.is_file():
        try:
            metadata = json.loads(manifest_path.read_text())
        except (OSError, json.JSONDecodeError) as error:
            print(f'Ignoring unreadable pretrained classifier manifest: {error}')
    model_metadata = metadata.get('model', {})
    imputation = (metadata.get('imputation') or {}).get('method')
    imputation = imputation or metadata.get('missing_data_method') or 'ffill'
    artifact_metadata = {
        'family': CLASSIFIER_FAMILY,
        'family_base_label': FAMILY_LABEL[CLASSIFIER_FAMILY],
        'family_label': FAMILY_LABEL[CLASSIFIER_FAMILY],
        'head_type': 'not_applicable',
        'head_label': HEAD_LABEL['not_applicable'],
        'training_regime': 'pretrained_classifier',
        'training_regime_label': TRAINING_REGIME_LABEL['pretrained_classifier'],
        'imputation': imputation,
        'imputation_label': IMPUTATION_LABEL.get(imputation, imputation),
        'model_type': 'inception-classifier',
        'model_label': MODEL_LABEL['inception-classifier'],
        'model_configuration': 'pretrained_inception_last_hour',
        'model_configuration_label': 'last-hour pretrained checkpoint',
        'run_id': 'pretrained_inception_last_hour',
    }
    if model_metadata:
        print(f'Including pretrained classifier: {model_metadata.get("model_class", "Inception_classifier")}')
    frames = {}
    for suffix in ('_landmark_metrics.csv', '_landmark_predictions.csv'):
        path = PRETRAINED_CLASSIFIER_RESULT_DIR / f'{pretrained_classifier_stem}{suffix}'
        if not path.is_file():
            print(f'Pretrained classifier artifact not found: {path}')
            frames[suffix] = empty
            continue
        try:
            frame = pd.read_csv(path)
        except (OSError, pd.errors.EmptyDataError) as error:
            print(f'Skipping pretrained classifier artifact {path.name}: {error}')
            frames[suffix] = empty
            continue
        for key, value in artifact_metadata.items():
            frame[key] = value
        frame['configuration_label'] = frame['model_label'] + ' — ' + frame['imputation_label']
        frame['source_file'] = path.name
        frame['source_path'] = str(path)
        frame['source_result_dir'] = str(PRETRAINED_CLASSIFIER_RESULT_DIR)
        frame['source_relative_path'] = path.relative_to(REPO_ROOT).as_posix()
        frames[suffix] = frame
    metrics = frames.get('_landmark_metrics.csv', empty)
    predictions = frames.get('_landmark_predictions.csv', empty)
    return metrics, predictions

def load_baseline_metrics(result_dir: Path) -> pd.DataFrame:
    # Baselines are intentionally loaded after the time-to-event artifacts
    # have been filtered and model selection has been applied. They have no
    # histories or prediction panels in this result directory.
    if not result_dir.is_dir():
        print(f'Baseline result directory not found: {result_dir}')
        return pd.DataFrame()
    frames = []
    for manifest_path in sorted(result_dir.glob('*_manifest.json')):
        model_id = manifest_path.stem.removesuffix('_manifest')
        family = {'ctg_fri_cox': 'baseline_ctg_only_fri_cox', 'xgboost_ctg': 'baseline_xgboost_figo_ctg_cox'}.get(model_id, f'baseline_{model_id}')
        metric_path = manifest_path.with_name(
            manifest_path.name.replace('_manifest.json', '_landmark_metrics.csv')
        )
        if not metric_path.is_file():
            print(f'Skipping baseline without metrics: {manifest_path.name}')
            continue
        try:
            metadata = json.loads(manifest_path.read_text())
            frame = pd.read_csv(metric_path)
        except (OSError, json.JSONDecodeError, pd.errors.EmptyDataError) as error:
            print(f'Skipping baseline {manifest_path.name}: {error}')
            continue
        family_label = FAMILY_LABEL.get(
            family, metadata.get('model', model_id)
        )
        frame['family'] = family
        frame['family_base_label'] = family_label
        frame['family_label'] = family_label
        frame['head_type'] = 'not_applicable'
        frame['head_label'] = HEAD_LABEL['not_applicable']
        frame['training_regime'] = 'not_applicable'
        frame['training_regime_label'] = TRAINING_REGIME_LABEL['not_applicable']
        frame['imputation'] = 'not_applicable'
        frame['imputation_label'] = 'Not applicable'
        frame['model_type'] = family
        frame['model_label'] = family_label
        frame['model_configuration'] = family
        frame['model_configuration_label'] = family_label
        frame['configuration_label'] = family_label
        frame['run_id'] = family
        frame['source_file'] = metric_path.name
        frame['source_path'] = str(metric_path)
        frame['source_relative_path'] = metric_path.relative_to(REPO_ROOT).as_posix()
        # Baselines are a single fixed configuration, so they occupy one
        # comparison slot rather than being filtered by TTE model settings.
        frame['comparison_value'] = 'Baseline'
        frame['comparison_label'] = family_label + ' — Baseline'
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_baseline_predictions(result_dir: Path) -> pd.DataFrame:
    frames = []
    for manifest_path in sorted(result_dir.glob('*_manifest.json')):
        model_id = manifest_path.stem.removesuffix('_manifest')
        family = {'ctg_fri_cox': 'baseline_ctg_only_fri_cox', 'xgboost_ctg': 'baseline_xgboost_figo_ctg_cox'}.get(model_id, f'baseline_{model_id}')
        prediction_path = manifest_path.with_name(f'{model_id}_landmark_predictions.csv')
        if not prediction_path.is_file():
            print(f'Skipping baseline predictions without artifact: {manifest_path.name}')
            continue
        try:
            metadata = json.loads(manifest_path.read_text())
            frame = pd.read_csv(prediction_path)
        except (OSError, json.JSONDecodeError, pd.errors.EmptyDataError) as error:
            print(f'Skipping baseline predictions {manifest_path.name}: {error}')
            continue
        family_label = FAMILY_LABEL.get(family, metadata.get('model', model_id))
        frame['family'] = family
        frame['family_base_label'] = family_label
        frame['family_label'] = family_label
        frame['head_type'] = 'not_applicable'
        frame['head_label'] = HEAD_LABEL['not_applicable']
        frame['training_regime'] = 'not_applicable'
        frame['training_regime_label'] = TRAINING_REGIME_LABEL['not_applicable']
        frame['imputation'] = 'not_applicable'
        frame['imputation_label'] = 'Not applicable'
        frame['model_type'] = family
        frame['model_label'] = family_label
        frame['model_configuration'] = family
        frame['model_configuration_label'] = family_label
        frame['configuration_label'] = family_label
        frame['run_id'] = family
        frame['source_file'] = prediction_path.name
        frame['source_path'] = str(prediction_path)
        frame['source_relative_path'] = prediction_path.relative_to(REPO_ROOT).as_posix()
        frame['comparison_value'] = 'Baseline'
        frame['comparison_label'] = family_label + ' — Baseline'
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _model_selection_candidates(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    candidates = frame
    if MODEL_TYPE_FILTER is not None:
        candidates = candidates.loc[candidates['model_type'].eq(MODEL_TYPE_FILTER)]
    if IMPUTATION_FILTER is not None:
        candidates = candidates.loc[candidates['imputation'].eq(IMPUTATION_FILTER)]
    return candidates

def _model_selection_family_columns(frame: pd.DataFrame) -> list[str]:
    missing = [column for column in MODEL_SELECTION_FAMILY_COLUMNS if column not in frame.columns]
    if missing:
        raise KeyError(
            f'MODEL_SELECTION_FAMILY_COLUMNS contains unavailable columns: {missing}. '
            f'Available columns include: {sorted(frame.columns.tolist())}'
        )
    return list(MODEL_SELECTION_FAMILY_COLUMNS)

def _model_selection_family_key(frame: pd.DataFrame, family_columns: list[str]) -> pd.Series:
    # A string key keeps the grouping explicit while the original columns
    # remain available for readable selection output and merging.
    return frame[family_columns].fillna('Missing').astype(str).agg(' — '.join, axis=1)

def choose_best_model_configurations(frame: pd.DataFrame) -> pd.DataFrame | None:
    if MODEL_SELECTION != 'best in family' or frame.empty:
        return None
    candidates = _model_selection_candidates(frame).copy()
    if candidates.empty:
        raise ValueError(
            'Model selection produced no candidates after MODEL_TYPE_FILTER/IMPUTATION_FILTER.'
        )
    available_pairs = set(
        candidates[['target', 'metric']].dropna().itertuples(index=False, name=None)
    )
    if (MODEL_SELECTION_TARGET, MODEL_SELECTION_METRIC) not in available_pairs:
        available_metrics = sorted(
            candidates.loc[candidates['target'].eq(MODEL_SELECTION_TARGET), 'metric']
            .dropna().unique()
        )
        raise ValueError(
            f"No rows found for MODEL_SELECTION_TARGET={MODEL_SELECTION_TARGET!r} and "
            f"MODEL_SELECTION_METRIC={MODEL_SELECTION_METRIC!r}. Available metrics for this "
            f"target: {available_metrics}"
        )
    selection_metrics = [MODEL_SELECTION_METRIC]
    if INCLUDE_END_LANDMARKS_IN_SELECTION and 'auroc_end_landmark' not in selection_metrics:
        selection_metrics.append('auroc_end_landmark')
    family_columns = _model_selection_family_columns(candidates)
    # The candidate configuration is family + head + sampling regime +
    # model_configuration (which contains depth/kernel metadata). This keeps
    # sampling and architecture settings available for selection even when
    # they are not part of the family grouping.
    selection_identity_columns = list(dict.fromkeys(
        family_columns + ['family', 'head_type', 'training_regime', 'model_configuration']
    ))
    configuration_columns = selection_identity_columns + ['model_configuration_label']
    configuration_group_columns = selection_identity_columns
    all_configurations = candidates[configuration_columns].drop_duplicates()
    # Keep landmark_seconds_since_start out of the grouping keys: each
    # candidate's selection_score is the reduced value across all landmarks
    # reported for the configured target and metric.
    score_rows = candidates.loc[
        candidates['split'].eq(MODEL_SELECTION_SPLIT)
        & candidates['target'].eq(MODEL_SELECTION_TARGET)
        & candidates['metric'].isin(selection_metrics)
        & candidates['value'].notna()
    ].copy()
    if score_rows.empty and MODEL_SELECTION_SPLIT != SPLIT:
        # If validation reports are unavailable, use the displayed split.
        score_rows = candidates.loc[
            candidates['split'].eq(SPLIT)
            & candidates['target'].eq(MODEL_SELECTION_TARGET)
            & candidates['metric'].isin(selection_metrics)
            & candidates['value'].notna()
        ].copy()
    if score_rows.empty:
        raise ValueError(
            f'No usable model-selection rows found for {MODEL_SELECTION_TARGET!r} / '
            f'{MODEL_SELECTION_METRIC!r} on {MODEL_SELECTION_SPLIT!r} or {SPLIT!r}.'
        )
    scores = score_rows.groupby(configuration_group_columns, as_index=False).agg(
        selection_score=('value', MODEL_SELECTION_REDUCTION), selection_rows=('value', 'count'),
        selection_landmarks=('landmark_seconds_since_start', 'nunique'),
    )
    chosen = all_configurations.merge(
        scores, on=configuration_group_columns, how='left'
    )
    chosen = chosen.loc[chosen['selection_score'].notna()].copy()
    if chosen.empty:
        raise ValueError('No model configurations have a finite selection score.')
    chosen['_score_for_sorting'] = chosen['selection_score']
    chosen['_model_selection_family'] = _model_selection_family_key(chosen, family_columns)
    chosen = (
        chosen.sort_values(
            ['_model_selection_family', '_score_for_sorting', 'model_configuration_label'],
            ascending=[True, False, True],
        )
        .groupby('_model_selection_family', as_index=False, sort=False)
        .head(1)
        .drop(columns=['_score_for_sorting', '_model_selection_family'])
        .reset_index(drop=True)
    )
    print(
        f"Model selection: best configuration per family using {selection_metrics} "
        f"({MODEL_SELECTION_REDUCTION}) / "
        f"{MODEL_SELECTION_TARGET} ({MODEL_SELECTION_SPLIT if not score_rows.empty else SPLIT})"
    )
    print(chosen[configuration_columns + ['selection_score', 'selection_rows', 'selection_landmarks']].to_string(index=False))
    return chosen[selection_identity_columns + ['model_configuration_label', 'selection_score']]

def select_comparison_subset(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    if MODEL_TYPE_FILTER is not None:
        frame = frame.loc[frame['model_type'].eq(MODEL_TYPE_FILTER)].copy()
    if IMPUTATION_FILTER is not None:
        frame = frame.loc[frame['imputation'].eq(IMPUTATION_FILTER)].copy()
    if MODEL_SELECTION == 'best in family' and SELECTED_MODEL_CONFIGURATIONS is not None:
        selection_merge_columns = list(dict.fromkeys(
            MODEL_SELECTION_FAMILY_COLUMNS + ['family', 'head_type', 'training_regime', 'model_configuration']
        ))
        frame = frame.merge(
            SELECTED_MODEL_CONFIGURATIONS[selection_merge_columns],
            on=selection_merge_columns, how='inner',
        )
    base_comparison = (
        frame[COMPARISON_COLUMN].astype(str)
        if COMPARISON_COLUMN is not None
        else pd.Series('Selected configuration', index=frame.index)
    )
    if MODEL_SELECTION == 'all':
        base_comparison = base_comparison + ' — ' + frame['model_configuration_label']
    frame['comparison_value'] = base_comparison + ' — ' + frame['training_regime_label']
    frame['comparison_label'] = frame['family_label'] + ' — ' + frame['comparison_value']
    return frame

def _resolve_checkpoint(metadata: dict, source_result_dir: Path) -> Path | None:
    # Check both current top-level metadata and older manifests that stored
    # the checkpoint under model.checkpoint.
    model_metadata = metadata.get('model') if isinstance(metadata.get('model'), dict) else {}
    checkpoint_values = [
        metadata.get('checkpoint'),
        metadata.get('source_checkpoint'),
        model_metadata.get('checkpoint'),
        model_metadata.get('source_checkpoint'),
        metadata.get('model_pickle'),
    ]
    checkpoint_values = list(dict.fromkeys(value for value in checkpoint_values if value))

    def is_packaged_input(candidate: Path) -> bool:
        try:
            candidate.resolve().relative_to(FINAL_SELECTION_DIR.resolve())
            return True
        except ValueError:
            return False

    for raw_value in checkpoint_values:
        raw_path = Path(str(raw_value)).expanduser()
        candidates = [raw_path] if raw_path.is_absolute() else [
            REPO_ROOT / raw_path,
            source_result_dir / raw_path,
            source_result_dir.parent / raw_path,
            source_result_dir / 'checkpoints' / raw_path.name,
            source_result_dir.parent / 'checkpoints' / raw_path.name,
        ]
        seen = set()
        for candidate in candidates:
            candidate = candidate.resolve()
            if candidate in seen or is_packaged_input(candidate):
                continue
            seen.add(candidate)
            if candidate.is_file():
                return candidate
    return None

def _package_artifact(src: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, destination)

def package_final_model_selection(
    selected_configurations: pd.DataFrame | None,
    selected_metrics: pd.DataFrame,
    selected_predictions: pd.DataFrame,
    selected_histories: pd.DataFrame,
) -> None:
    """Create a portable archive of every artifact in the selected configurations."""
    if selected_configurations is None:
        selected_configurations = pd.DataFrame()
    if selected_configurations.empty and selected_metrics.empty and selected_predictions.empty:
        print('No selected artifacts; final_model_selection was not created.')
        return
    artifact_root = FINAL_SELECTION_DIR / 'artifacts'
    checkpoint_root = FINAL_SELECTION_DIR / 'checkpoints'
    selection_root = FINAL_SELECTION_DIR / 'selection'
    artifact_root.mkdir(parents=True, exist_ok=True)
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    selection_root.mkdir(parents=True, exist_ok=True)
    (checkpoint_root / 'README.txt').write_text(
        'When run on the training machine, selected .pt checkpoints and baseline .pkl bundles are copied here.\n'
        'A missing model artifact is recorded in selection/selected_artifacts.csv and each packaged manifest.\n',
        encoding='utf-8',
    )

    artifact_frames = {
        'history': selected_histories,
        'landmark_metrics': selected_metrics,
        'landmark_predictions': selected_predictions,
    }
    artifact_records = []
    copied_manifest_sources = set()
    copied_checkpoints = {}
    checkpoint_statuses = []
    for artifact_type, frame in artifact_frames.items():
        if frame.empty or 'source_path' not in frame.columns:
            continue
        source_rows = frame[['source_path', 'source_relative_path']].drop_duplicates()
        for source_path_text, source_relative_path in source_rows.itertuples(index=False):
            source_path = Path(source_path_text)
            if not source_path.is_file():
                artifact_records.append({
                    'artifact_type': artifact_type,
                    'source_path': str(source_relative_path),
                    'packaged_path': None,
                    'status': 'missing',
                })
                continue
            destination = artifact_root / source_relative_path
            _package_artifact(source_path, destination)
            artifact_records.append({
                'artifact_type': artifact_type,
                'source_path': str(source_relative_path),
                'packaged_path': destination.relative_to(FINAL_SELECTION_DIR).as_posix(),
                'status': 'copied',
            })
            manifest_source = source_path.parent / f'{artifact_stem(source_path)}.json'
            if not manifest_source.is_file():
                manifest_source = source_path.parent / f'{artifact_stem(source_path)}_manifest.json'
            if manifest_source.is_file() and manifest_source not in copied_manifest_sources:
                copied_manifest_sources.add(manifest_source)
                manifest_relative = manifest_source.relative_to(REPO_ROOT)
                manifest_destination = artifact_root / manifest_relative
                metadata = json.loads(manifest_source.read_text())
                source_result_dir = manifest_source.parent
                checkpoint_source = _resolve_checkpoint(metadata, source_result_dir)
                checkpoint_destination = None
                if checkpoint_source is not None:
                    checkpoint_destination = checkpoint_root / source_result_dir.name / checkpoint_source.name
                    _package_artifact(checkpoint_source, checkpoint_destination)
                    copied_checkpoints[str(checkpoint_source)] = checkpoint_destination
                    checkpoint_statuses.append('copied')
                    for sibling in sorted(checkpoint_source.parent.glob(f'{checkpoint_source.stem}*')):
                        if sibling.is_file() and sibling != checkpoint_source:
                            _package_artifact(sibling, checkpoint_destination.parent / sibling.name)
                    # The evaluator also accepts a legacy config one
                    # directory above the checkpoint; normalize it beside
                    # the copied checkpoint in the portable archive.
                    for config_candidate in (
                        checkpoint_source.with_name(checkpoint_source.stem + '_training_config.json'),
                        checkpoint_source.parent.parent / (checkpoint_source.stem + '_training_config.json'),
                    ):
                        if config_candidate.is_file():
                            _package_artifact(config_candidate, checkpoint_destination.parent / config_candidate.name)
                if checkpoint_source is None:
                    checkpoint_statuses.append('missing')
                original_checkpoint = metadata.get('checkpoint')
                original_source_checkpoint = metadata.get('source_checkpoint')
                original_model_pickle = metadata.get('model_pickle')
                metadata['original_checkpoint'] = original_checkpoint
                metadata['original_source_checkpoint'] = original_source_checkpoint
                metadata['original_model_pickle'] = original_model_pickle
                metadata['checkpoint_status'] = 'copied' if checkpoint_destination else 'missing'
                if checkpoint_destination:
                    packaged_checkpoint = checkpoint_destination.relative_to(FINAL_SELECTION_DIR).as_posix()
                    metadata['checkpoint'] = packaged_checkpoint
                    metadata['source_checkpoint'] = packaged_checkpoint
                    if original_model_pickle:
                        metadata['model_pickle'] = packaged_checkpoint
                else:
                    metadata['checkpoint'] = original_checkpoint
                    metadata['source_checkpoint'] = original_source_checkpoint
                for key, source_artifact in (
                    ('landmark_metrics', source_result_dir / f'{artifact_stem(source_path)}_landmark_metrics.csv'),
                    ('landmark_predictions', source_result_dir / f'{artifact_stem(source_path)}_landmark_predictions.csv'),
                ):
                    if source_artifact is not None:
                        metadata[key] = (artifact_root / source_artifact.relative_to(REPO_ROOT)).relative_to(FINAL_SELECTION_DIR).as_posix()
                metadata['packaged_manifest'] = manifest_destination.relative_to(FINAL_SELECTION_DIR).as_posix()
                metadata['packaged_from'] = manifest_relative.as_posix()
                manifest_destination.parent.mkdir(parents=True, exist_ok=True)
                manifest_destination.write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

    selected_configurations.to_csv(selection_root / 'selected_configurations.csv', index=False)
    pd.DataFrame(artifact_records).drop_duplicates().to_csv(
        selection_root / 'selected_artifacts.csv', index=False
    )
    selection_manifest = {
        'result_directories': [path.relative_to(REPO_ROOT).as_posix() for path in RESULT_DIRS],
        'selection': {
            'split': MODEL_SELECTION_SPLIT,
            'target': MODEL_SELECTION_TARGET,
            'metric': MODEL_SELECTION_METRIC,
            'reduction': MODEL_SELECTION_REDUCTION,
            'family_columns': MODEL_SELECTION_FAMILY_COLUMNS,
        },
        'selected_configurations': selected_configurations.to_dict(orient='records'),
        'artifact_root': 'artifacts',
        'checkpoint_root': 'checkpoints',
        'selection_root': 'selection',
        'checkpoint_status_counts': pd.Series(checkpoint_statuses).value_counts().to_dict(),
    }
    (FINAL_SELECTION_DIR / 'selection_manifest.json').write_text(
        json.dumps(selection_manifest, indent=2, default=str) + '\n', encoding='utf-8'
    )
    print(f'Wrote portable selected-model archive to {FINAL_SELECTION_DIR}')
    print(f'Copied source artifacts: {sum(record["status"] == "copied" for record in artifact_records)}')
    print(f'Resolved checkpoints: {len(copied_checkpoints)}; unresolved checkpoints remain documented in packaged manifests.')

raw_histories = load_artifacts('_history.csv')
raw_metrics = load_artifacts('_landmark_metrics.csv')
raw_predictions = load_artifacts('_landmark_predictions.csv')
pretrained_classifier_metrics, pretrained_classifier_predictions = load_pretrained_classifier_artifacts()
if not pretrained_classifier_metrics.empty:
    raw_metrics = pd.concat([raw_metrics, pretrained_classifier_metrics], ignore_index=True)
if not pretrained_classifier_predictions.empty:
    raw_predictions = pd.concat([raw_predictions, pretrained_classifier_predictions], ignore_index=True)
baseline_metrics = load_baseline_metrics(BASELINE_RESULT_DIR)
baseline_predictions = load_baseline_predictions(BASELINE_RESULT_DIR)
SELECTED_MODEL_CONFIGURATIONS = choose_best_model_configurations(raw_metrics)
# Everything downstream starts from the selected result artifacts. Baselines
# have fixed configurations, so they are appended unchanged after selection.
histories = select_comparison_subset(raw_histories)
selected_metrics = select_comparison_subset(raw_metrics)
predictions = select_comparison_subset(raw_predictions)
package_final_model_selection(
    SELECTED_MODEL_CONFIGURATIONS,
    pd.concat([selected_metrics, baseline_metrics], ignore_index=True) if not baseline_metrics.empty else selected_metrics,
    pd.concat([predictions, baseline_predictions], ignore_index=True) if not baseline_predictions.empty else predictions,
    histories,
)
metrics = selected_metrics.copy()
if not baseline_metrics.empty:
    metrics = pd.concat([metrics, baseline_metrics], ignore_index=True)
    print(f'Loaded baseline metric files: {baseline_metrics.source_file.nunique()}')
if not baseline_predictions.empty:
    predictions = pd.concat([predictions, baseline_predictions], ignore_index=True)
    print(f'Loaded baseline prediction files: {baseline_predictions.source_file.nunique()}')
if not histories.empty:
    histories['epoch'] = histories.groupby('source_file').cumcount() + 1

print(f'History files: {histories.source_file.nunique() if not histories.empty else 0}')
print(f'Completed metric files: {metrics.source_file.nunique() if not metrics.empty else 0}')
print(f'Completed prediction files: {predictions.source_file.nunique() if not predictions.empty else 0}')